# Train A2C

A2C improves Actor-Critic by learning from multi-step rollouts and generalized advantage estimates (GAE):

$$\delta_t=r_{t+1}+\gamma V_\phi(s_{t+1})-V_\phi(s_t),\qquad A_t=\sum_{l\geq0}(\gamma\lambda)^l\delta_{t+l}.$$

Here $\delta_t$ is the TD error, $A_t$ the advantage estimate, $V_\phi$ the critic, $\gamma$ the discount factor, and $\lambda$ the GAE parameter. This notebook trains A2C on `CartPole-v1` with a categorical policy, advantage normalization, and entropy regularization.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import A2C, A2CConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "CartPole-v1"

In [ ]:
env = gym.make(ENV_ID)
config = A2CConfig(
    learning_rate=7e-4,
    value_learning_rate=7e-4,
    gamma=0.99,
    n_steps=32,
    gae_lambda=0.95,
    normalize_advantage=True,
    entropy_coefficient=1e-3,
)

agent = A2C(env, config=config, device="cpu")
agent.learn(total_timesteps=20_000)
env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(20, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"A2C training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human")
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")